# S4.6 - Window specification
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** A window keeps every row and adds a calculation over related rows: partitionBy, orderBy, frame

In [0]:
# ============================================================
# S4.6 - Cell 2
# Window Specification:
# partitionBy, orderBy, ROWS frame, RANGE frame
#
# Goal:
# 1. Compare groupBy (collapses rows) with a window (keeps rows).
# 2. Understand what partitionBy controls.
# 3. Understand what orderBy controls inside a window.
# 4. Understand the default ordered window frame.
# 5. Compare RANGE and ROWS when ORDER BY values contain ties.
# 6. Understand why ROWS may need a deterministic tie-breaker.
# 7. Validate the window specification on samples.tpch.
# 8. Observe the scalability risk of omitting partitionBy.
#
# Scope:
# Window specification only.
#
# Deferred:
# - row_number / rank / dense_rank / ntile -> S4.7
# - lag / lead / first / last              -> S4.8
# - running sum / rolling average          -> S4.9
# - rolling 7-day RetailPulse exercise     -> S4.10
#
# Note:
# sum() is used here only to make window-frame behaviour
# visible. Running/rolling calculations are studied fully
# in S4.9.
# ============================================================

import io
import contextlib

import pyspark.sql.functions as F
from pyspark.sql import Window


print("=== S4.6: WINDOW SPECIFICATION ===")
print()


# ============================================================
# Step 1 - Controlled Source Data
#
# Customer 1 deliberately has TWO rows for day = 2.
#
# This lets us observe how:
#
# RANGE
# and
# ROWS
#
# behave when ORDER BY values are tied.
# ============================================================

df = spark.createDataFrame(
    [
        (1, 1, 10),
        (1, 2, 20),
        (1, 2, 5),
        (1, 3, 30),
        (2, 1, 100),
        (2, 2, 50)
    ],
    """
    cust INT,
    day INT,
    amount INT
    """
)

source_count = (df.count())

print("=== STEP 1: SOURCE ===")

df.orderBy(
    "cust",
    "day",
    "amount"
).show()


print(
    f"RESULT source rows: "
    f"{source_count}")


In [0]:
# ============================================================
# Step 2 - groupBy vs Window
#
# Learning goal:
#
# Show clearly that:
#
# groupBy(...)
# → collapses many rows into one row per group
#
# Window.partitionBy(...)
# → calculates across the group
#   but keeps every original row
#
#
# Source:
#
# Customer 1:
# amounts = 10, 20, 5, 30
# total   = 65
#
# Customer 2:
# amounts = 100, 50
# total   = 150
#
#
# Expected:
#
# groupBy rows = 2
#
# Window rows = 6
#
#
# Important:
#
# Window has NO orderBy here.
#
# Therefore the SUM sees the COMPLETE customer partition.
# Every row belonging to the same customer receives the
# same customer total.
# ============================================================
    
# ------------------------------------------------------------
# GROUP BY
#
# Many source rows
#       ↓
# group by customer
#       ↓
# one result row per customer
# ------------------------------------------------------------

grouped = (
    df
    .groupBy("cust")
    .agg(F.sum("amount").alias("total")))

# ------------------------------------------------------------
# WINDOW
#
# Partition rows by customer.
#
# No orderBy:
# the aggregate sees the complete customer partition.
#
# Original rows are NOT collapsed.
# ------------------------------------------------------------

w_partition = (Window.partitionBy("cust"))

window_total = (
    df
    .withColumn(
        "partition_amounts",
        F.sort_array(
            F.collect_list("amount").over(
                w_partition)))
    .withColumn(
        "cust_total",
        F.sum("amount").over(
            w_partition)))

# ------------------------------------------------------------
# Row counts
# ------------------------------------------------------------

grouped_count = (grouped.count())

window_count = (window_total.count())

# ------------------------------------------------------------
# Validation values
# ------------------------------------------------------------

totals = sorted(
    {
        (
            r["cust"],
            r["cust_total"]
        )
        for r in window_total.collect()})

print()
print("=== STEP 2: groupBy vs WINDOW ===")

# ============================================================
# Part A - Show GROUP BY result
# ============================================================

print()
print("=== GROUP BY ===")

print(
    """
groupBy collapses rows.

Customer rows:
many
  ↓
aggregate
  ↓
one row per customer
"""
)

grouped.orderBy("cust").show()

print(f"RESULT groupBy rows: "f"{grouped_count}")

# ============================================================
# Part B - Show WINDOW result
# ============================================================

print()
print("=== WINDOW ===")

print(
    """
Window does NOT collapse the original rows.

partitionBy("cust")

means:

each customer forms its own logical partition.

Because there is NO orderBy here,
SUM(amount) uses the complete customer partition.

Therefore every row for the same customer receives
the same customer total.
"""
)

learning_rows = (
    window_total
    .orderBy("cust","amount").collect())

for r in learning_rows:

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current amount={r['amount']} | "
        f"partition amounts={r['partition_amounts']} | "
        f"customer total={r['cust_total']}")

# ============================================================
# Part C - Compare row counts directly
# ============================================================

print()
print("=== KEY DIFFERENCE ===")

print(f"Source rows   : {source_count}")

print(f"groupBy rows  : {grouped_count}")

print(f"Window rows   : {window_count}")

print(
    """
groupBy:

6 source rows
      ↓
2 result rows


Window:

6 source rows
      ↓
6 result rows remain
+
aggregate value added to each row
"""
)

# ============================================================
# Part D - Final validation
# ============================================================

print("RESULT totals by customer:",totals)

In [0]:
# ============================================================
# Step 3 - Default Frame With orderBy
#
# Window:
#
# partitionBy("cust")
# orderBy("day")
#
# For an aggregate window with ORDER BY, the default
# ordered frame behaves as:
#
# RANGE BETWEEN
# UNBOUNDED PRECEDING
# AND CURRENT ROW
#
#
# RANGE works using ORDER BY VALUES.
#
# Therefore rows sharing the same ORDER BY value are peers.
#
#
# Customer 1 prediction:
#
# day 1:
#
# frame:
# (day=1, amount=10)
#
# running = 10
#
#
# day 2:
#
# There are TWO rows where day = 2.
#
# Because RANGE works by ORDER BY value,
# both day=2 rows belong to the same peer group.
#
# frame:
#
# (day=1, amount=10)
# (day=2, amount=5)
# (day=2, amount=20)
#
# running:
#
# 10 + 5 + 20 = 35
#
# Therefore BOTH day=2 rows receive:
#
# running = 35
#
#
# day 3:
#
# frame contains all customer-1 rows.
#
# running:
#
# 10 + 5 + 20 + 30 = 65
#
#
# Learning output:
#
# current row
# +
# rows inside its frame
# +
# resulting sum
# ============================================================


w_default = (
    Window
    .partitionBy("cust")
    .orderBy("day")
)


default_running = (
    df

    # --------------------------------------------------------
    # Show the rows participating in the current frame.
    #
    # struct(day, amount)
    # makes each frame member easy to understand.
    #
    # sort_array is used only to make the learning output
    # easier to read consistently.
    # --------------------------------------------------------

    .withColumn(
        "frame_rows",
        F.sort_array(
            F.collect_list(
                F.struct(
                    "day",
                    "amount"
                )
            ).over(
                w_default
            )
        )
    )

    # --------------------------------------------------------
    # Actual window calculation.
    # --------------------------------------------------------

    .withColumn(
        "running",
        F.sum(
            "amount"
        ).over(
            w_default
        )
    )
)


# ------------------------------------------------------------
# Keep the original compact values.
#
# We will use these again in Step 4 to prove that:
#
# default ordered frame
# ==
# explicit RANGE frame
# ------------------------------------------------------------

default_values = sorted(
    [
        (
            r["cust"],
            r["day"],
            r["amount"],
            r["running"]
        )
        for r in default_running.collect()
    ]
)


# ============================================================
# Learner-Friendly Output
# ============================================================

print()
print(
    "=== STEP 3: DEFAULT ORDERED FRAME ==="
)


print(
    """
WINDOW

partitionBy("cust")
orderBy("day")

No frame was written explicitly.

For this aggregate window, Spark behaves like:

RANGE BETWEEN
UNBOUNDED PRECEDING
AND CURRENT ROW


HOW TO READ THE OUTPUT

current
→ the row Spark is calculating

frame
→ rows included in that calculation

sum
→ SUM(amount) across those frame rows
"""
)


learning_rows = (
    default_running
    .orderBy(
        "cust",
        "day",
        "amount"
    )
    .collect()
)


for r in learning_rows:

    frame = [
        (
            x["day"],
            x["amount"]
        )
        for x in r["frame_rows"]
    ]

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"frame={frame} | "
        f"sum={r['running']}"
    )


# ============================================================
# Highlight the Important RANGE Behaviour
# ============================================================

print()
print(
    "=== KEY OBSERVATION: TIED ORDER BY VALUES ==="
)


print(
    """
Customer 1 contains:

(day=2, amount=5)
(day=2, amount=20)

The window is ordered only by:

day

Therefore both rows have the SAME ORDER BY value:

day = 2


With the default RANGE frame:

both rows are peers.


For BOTH day=2 rows, the frame is:

(day=1, amount=10)
(day=2, amount=5)
(day=2, amount=20)


Calculation:

10 + 5 + 20 = 35


Therefore:

(day=2, amount=5)  -> running = 35
(day=2, amount=20) -> running = 35
"""
)

In [0]:
# ============================================================
# Step 4 - Explicit RANGE Frame
#
# Now write the default ordered aggregate frame explicitly:
#
# RANGE BETWEEN
# UNBOUNDED PRECEDING
# AND CURRENT ROW
#
# Prediction:
#
# Step 3
# and
# Step 4
#
# should produce the SAME results.
#
# Learning goal:
# Show not only the final running total, but also which
# amounts are included in the current RANGE frame.
# ============================================================

w_range = (
    Window
    .partitionBy("cust")
    .orderBy("day")
    .rangeBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


range_running = (
    df
    .withColumn(
        "frame_amounts",
        F.sort_array(
            F.collect_list("amount").over(
                w_range
            )
        )
    )
    .withColumn(
        "running",
        F.sum("amount").over(
            w_range
        )
    )
)


# ------------------------------------------------------------
# Keep the original values for validation against Step 3.
# ------------------------------------------------------------

range_values = sorted(
    [
        (
            r["cust"],
            r["day"],
            r["amount"],
            r["running"]
        )
        for r in range_running.collect()
    ]
)


# ------------------------------------------------------------
# Learner-friendly output
# ------------------------------------------------------------

print()
print(
    "=== STEP 4: EXPLICIT RANGE FRAME ==="
)

print(
    """
How to read this:

Window:
partitionBy("cust")
orderBy("day")
RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW

RANGE works using the ORDER BY VALUE.

So for a current row where day = 2:

Spark includes all rows for the same customer
whose day is <= 2.

If multiple rows have day = 2,
they are peers and are included together.
"""
)


learning_rows = (
    range_running
    .orderBy(
        "cust",
        "day",
        "amount"
    )
    .collect()
)


for r in learning_rows:

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"frame={r['frame_amounts']} | "
        f"sum={r['running']}"
    )


# ------------------------------------------------------------
# Highlight the important tie example.
# ------------------------------------------------------------

print()
print(
    "=== KEY OBSERVATION ==="
)

print(
    """
Customer 1 has two rows where day = 2:

(day=2, amount=5)
(day=2, amount=20)

Because RANGE uses the ORDER BY value:

day = 2
day = 2

both rows belong to the same peer group.

Therefore both frames contain:

10 + 5 + 20 = 35

So both day=2 rows receive:

running = 35
"""
)


# ------------------------------------------------------------
# Validation:
# Step 3 default frame should equal explicit RANGE.
# ------------------------------------------------------------

print(
    "RESULT default == explicit RANGE:",
    default_values == range_values
)

In [0]:
# ============================================================
# Step 5 - Explicit ROWS Frame With the SAME orderBy
#
# IMPORTANT:
#
# We keep:
#
# orderBy("day")
#
# exactly the same as Step 4.
#
# The ONLY conceptual change is:
#
# RANGE
# →
# ROWS
#
#
# RANGE:
# works using ORDER BY values.
#
# ROWS:
# works using physical row positions inside the ordered
# partition.
#
#
# Therefore, when day = 2 appears twice:
#
# RANGE
# → both day=2 rows shared the same frame
#
# ROWS
# → the frame grows one row at a time
#
#
# BUT:
#
# orderBy("day")
#
# does NOT uniquely order the two day=2 rows.
#
# Spark may process:
#
# amount=5 first
#
# OR
#
# amount=20 first
#
#
# Therefore:
#
# Do NOT assert which tied row receives the intermediate
# running total.
#
# The important learning point is:
#
# the two tied rows can receive DIFFERENT running totals.
# ============================================================


w_rows_tied = (
    Window
    .partitionBy("cust")
    .orderBy("day")
    .rowsBetween(Window.unboundedPreceding,Window.currentRow))

# ------------------------------------------------------------
# Learning output:
#
# frame_rows shows which rows participate in the current
# ROWS frame.
#
# running shows SUM(amount) over that frame.
# ------------------------------------------------------------

rows_tied_running = (
    df
    .withColumn(
        "frame_rows",
        F.collect_list(F.struct("day","amount")).over(w_rows_tied))

    .withColumn("running",F.sum("amount").over(w_rows_tied)))

# ------------------------------------------------------------
# Compact values retained for later comparison.
#
# Do NOT use these values to assert which day=2 row came
# first because the tie is unresolved.
# ------------------------------------------------------------

rows_tied_values = sorted(
    [
        (
            r["cust"],
            r["day"],
            r["amount"],
            r["running"]
        )
        for r in rows_tied_running.collect()
    ]
)


print()
print("=== STEP 5: ROWS FRAME WITH TIED ORDER ===")


print(
    """
WINDOW

partitionBy("cust")
orderBy("day")

ROWS BETWEEN
UNBOUNDED PRECEDING
AND CURRENT ROW


HOW TO READ THIS

current
→ row currently being calculated

frame
→ physical rows included up to that row

sum
→ SUM(amount) across those rows


IMPORTANT

Customer 1 has two rows where:

day = 2

Because orderBy("day") does not uniquely order them,
Spark may process either tied row first.

Therefore the intermediate running total is not
deterministically attached to amount=5 or amount=20.
"""
)


# ------------------------------------------------------------
# Collect in execution-frame form first.
#
# We deliberately do not sort frame_rows because ROWS is
# teaching us about sequence.
# ------------------------------------------------------------

learning_rows = (
    rows_tied_running
    .orderBy(
        "cust",
        "day",
        "amount"
    )
    .collect()
)


for r in learning_rows:

    frame = [
        (
            x["day"],
            x["amount"]
        )
        for x in r["frame_rows"]
    ]

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"frame={frame} | "
        f"sum={r['running']}"
    )


# ============================================================
# Key Observation
# ============================================================

print()
print(
    "=== KEY OBSERVATION: ROWS vs RANGE ==="
)


print(
    """
In Step 4 with RANGE:

both day=2 rows used the SAME frame:

[(day=1, amount=10),
 (day=2, amount=5),
 (day=2, amount=20)]

Therefore both received:

running = 35


In this Step 5 with ROWS:

the frame grows one physical row at a time.

Conceptually it behaves like:

first row
→ [10]

first day=2 row
→ [10, one day=2 row]

second day=2 row
→ [10, both day=2 rows]

day=3
→ [10, both day=2 rows, 30]


So the tied day=2 rows can receive DIFFERENT
running totals.
"""
)


# ------------------------------------------------------------
# Learner checkpoint
# ------------------------------------------------------------

cust1_day2 = [
    r
    for r in rows_tied_running.collect()
    if (r["cust"] == 1 and r["day"] == 2)]

day2_running_values = sorted(
    [
        r["running"]
        for r in cust1_day2])

print("RESULT customer 1 day=2 running values:",day2_running_values)

print("RESULT tied rows received different running values:",len(set(day2_running_values)) == 2)

In [0]:
# ============================================================
# Step 6 - Deterministic ROWS Frame
#
# For ROWS, sequence matters.
#
# Therefore add a tie-breaker:
#
# orderBy(
#     day,
#     amount
# )
#
#
# In this controlled dataset:
#
# (day, amount)
#
# uniquely orders the rows inside each customer.
#
#
# Expected customer 1:
#
# day 1 amount 10
# → running 10
#
# day 2 amount 5
# → running 15
#
# day 2 amount 20
# → running 35
#
# day 3 amount 30
# → running 65
#
#
# Expected customer 2:
#
# day 1 amount 100
# → 100
#
# day 2 amount 50
# → 150
# ============================================================

w_rows_deterministic = (Window.partitionBy("cust").orderBy("day","amount")
    .rowsBetween(Window.unboundedPreceding,Window.currentRow))

rows_running = (
    df.withColumn("running",F.sum("amount").over(w_rows_deterministic)))

deterministic_values = sorted(
    [
        (
            r["cust"],
            r["day"],
            r["amount"],
            r["running"]
        )
        for r in rows_running.collect()])

print()
print("=== STEP 6: DETERMINISTIC ROWS FRAME ===")

for value in deterministic_values:

    print(
        "RESULT",
        value)

In [0]:
# ============================================================
# Step 7 - Controlled Validation
#
# Assertions are appropriate because this is a tiny,
# controlled dataset with known expected results.
# ============================================================

print()
print("=== STEP 7: CONTROLLED CHECKPOINT ===")

expected_totals = [
    (1, 65),
    (2, 150)
]

expected_deterministic = [
    (1, 1, 10, 10),
    (1, 2, 5, 15),
    (1, 2, 20, 35),
    (1, 3, 30, 65),
    (2, 1, 100, 100),
    (2, 2, 50, 150)
]


assert (source_count == 6)

assert (grouped_count == 2)

assert (window_count == 6)

assert (totals == expected_totals)

assert (default_values ==range_values)

assert (deterministic_values == expected_deterministic)

print("RESULT source rows                 :",source_count)

print("RESULT groupBy rows                :",grouped_count)

print("RESULT window rows                 :",window_count)

print("RESULT customer totals             :",totals)

print("RESULT default == explicit RANGE   :",default_values == range_values)

print("RESULT deterministic ROWS matched :",deterministic_values == expected_deterministic)

print("RESULT all controlled checks passed.")



In [0]:
# ============================================================
# Step 8 - Real Data: TPC-H
#
# Use the same three customers from our earlier exercises.
#
#
# Window specification:
#
# partitionBy:
# o_custkey
#
# orderBy:
# o_orderdate
# o_orderkey
#
# o_orderkey acts as a deterministic tie-breaker when
# multiple orders share the same date.
#
#
# Frame:
#
# ROWS BETWEEN
# UNBOUNDED PRECEDING
# AND CURRENT ROW
#
#
# We also calculate the FULL customer total using a
# partition-only window.
#
#
# On the final ordered row for each customer:
#
# running
# should equal
# customer_total
#
#
# We intentionally do NOT use row_number() here because
# ranking belongs to S4.7.
# ============================================================

orders = (
    spark.table("samples.tpch.orders")
    .filter(F.col("o_custkey").isin(721180,382414,299701))

    .select(
        "o_custkey",
        "o_orderkey",
        "o_orderdate",
        "o_totalprice"))

w_real_running = (
    Window.partitionBy("o_custkey")
    .orderBy("o_orderdate","o_orderkey")
    .rowsBetween(Window.unboundedPreceding,Window.currentRow))

w_real_total = (Window.partitionBy("o_custkey"))

real = (
    orders

    .withColumn("running",
        F.sum("o_totalprice").over(w_real_running))

    .withColumn(
        "customer_total",
        F.sum("o_totalprice").over(w_real_total)))

# ------------------------------------------------------------
# Only three customers are used here.
#
# Therefore collecting this small validation result is
# acceptable for this controlled learning exercise.
#
# Sort by exactly the same business order as the window.
# ------------------------------------------------------------

real_rows = sorted(
    real.collect(),
    key=lambda r: (
        r["o_custkey"],
        r["o_orderdate"],
        r["o_orderkey"]))

# ------------------------------------------------------------
# Keep the LAST ordered row encountered for each customer.
#
# This is only Python-side validation for three customers.
#
# We are NOT introducing row_number() yet.
# ------------------------------------------------------------

last_by_customer = {}

for r in real_rows:

    last_by_customer[
        r["o_custkey"]] = r

print()
print("=== STEP 8: REAL DATA CHECK ===")

real_checks = []

for cust in sorted(last_by_customer):

    r = (last_by_customer[cust])

    check = (r["running"] == r["customer_total"])

    real_checks.append(check)

    print(
        "RESULT",
        (
            cust,
            r["running"],
            r["customer_total"],
            check))

print(
    "RESULT all final running totals "
    "equal customer totals:",
    all(real_checks))

In [0]:
# ============================================================
# Step 9 - No partitionBy
#
# If partitionBy is omitted:
#
# Window.orderBy(...)
#
# treats the complete DataFrame as one logical window
# partition.
#
#
# For large datasets this can create a serious scalability
# problem because all rows may need to be processed together.
#
#
# Prediction:
#
# The physical plan should indicate a single-partition
# requirement.
#
# We test what THIS workspace actually shows instead of
# assuming.
# ============================================================

global_window = (Window.orderBy("day")
    .rowsBetween(Window.unboundedPreceding,Window.currentRow))

global_running = (df.withColumn("running",F.sum("amount").over(global_window)))

buf = io.StringIO()

with contextlib.redirect_stdout(buf):

    global_running.explain(mode="formatted")

plan = (buf.getvalue())

single_partition_seen = ("SinglePartition"in plan)

print()
print("=== STEP 9: NO partitionBy ===")

print("RESULT plan mentions SinglePartition:",single_partition_seen)

In [0]:
# ============================================================
# Step 10 - S4.6 Checkpoint
#
# These are the observations to copy into the final
# S4.6 Markdown note.
# ============================================================

print()
print("=== STEP 10: S4.6 CHECKPOINT ===")

print(f"Source rows                     : "f"{source_count}")

print(f"groupBy rows                    : "f"{grouped_count}")

print(f"Window rows                     : "f"{window_count}")

print(f"Customer totals  : "f"{totals}")

print(f"Default frame == explicit RANGE : "f"{default_values == range_values}")

print(f"Deterministic ROWS matched      : "f"{deterministic_values == expected_deterministic}")

print(f"TPC-H final running == total    : "f"{all(real_checks)}")

print(f"No-partition plan SinglePartition: "f"{single_partition_seen}"_

In [0]:

# ============================================================
# Step 11 - Final Learning Summary
# ============================================================

print()
print("=== FINAL LEARNING SUMMARY ===")

print("""
WINDOW SPECIFICATION

A window answers three questions:

1. WHICH ROWS BELONG TOGETHER?

partitionBy(...)

2. IN WHAT ORDER?

orderBy(...)

3. WHICH ROWS AROUND THE CURRENT ROW PARTICIPATE?

window frame

--------------------------------------------

groupBy

many source rows
      ↓
aggregate
      ↓
fewer result rows

Window

many source rows
      ↓
window calculation
      ↓
same source rows retained

--------------------------------------------

partitionBy("cust")

Creates an independent logical window partition
for each customer.

--------------------------------------------

orderBy("day")

Defines processing order inside each partition.

--------------------------------------------

WITHOUT orderBy

An aggregate window over:

Window.partitionBy(...)

uses the complete partition.

--------------------------------------------

WITH orderBy

The default ordered aggregate frame behaves as:

RANGE BETWEEN
UNBOUNDED PRECEDING
AND CURRENT ROW

--------------------------------------------

RANGE

Works from ORDER BY values.

Rows with the same ORDER BY value are peers.

Therefore tied rows can share the same
aggregate frame.

--------------------------------------------

ROWS

Works from row positions.

Rows tied on the ORDER BY value can therefore
receive different cumulative results.

--------------------------------------------

ROWS + TIES

If sequence matters and ORDER BY is not unique:

add a meaningful tie-breaker.

Example:

orderBy(business_date,transaction_id)

--------------------------------------------

NO partitionBy

The entire DataFrame can become one logical
window partition.

That may create a serious scalability problem
for large datasets.

--------------------------------------------

IMPORTANT SCOPE

We used SUM only to make frame behaviour visible.

Full running-sum and rolling-average patterns
are covered in:

S4.9

Ranking functions are covered in:

S4.7

lag / lead / first / last are covered in:

S4.8

--------------------------------------------

CORE RULE

Window specification:

partitionBy
→ grouping / logical partition

orderBy
→ sequence inside that partition

frame
→ which rows participate for the current row
""")

print("=== S4.6 CELL 2 COMPLETED ===")

# S4.6 — Key Takeaways: Window Specification

## 1. groupBy vs Window

`groupBy` changes the grain of the data:

```text
many source rows
→ aggregate
→ one row per group
```

A window keeps the original rows:

```text
many source rows
→ window calculation
→ same rows remain
→ calculated value added to each row
```

Observed:

```text
source rows  = 6
groupBy rows = 2
window rows  = 6
```

Customer totals:

```text
customer 1 = 65
customer 2 = 150
```

With:

```python
Window.partitionBy("cust")
```

and no `orderBy`, each row sees the complete customer partition.

Therefore every row for the same customer receives the same customer total.

---

## 2. Three Parts of a Window

```python
from pyspark.sql import Window

w = (
    Window
    .partitionBy("cust")
    .orderBy("day")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)
```

| Part | Question it answers |
|---|---|
| `partitionBy` | Which rows belong together? |
| `orderBy` | In what sequence are rows processed inside the partition? |
| frame | Which rows participate in the calculation for the current row? |

A useful mental model is:

```text
partitionBy
→ group

orderBy
→ sequence

frame
→ calculation boundary
```

---

## 3. Default Window Frames

### Without orderBy

For an aggregate window such as:

```python
Window.partitionBy("cust")
```

the default frame covers the complete partition:

```text
UNBOUNDED PRECEDING
to
UNBOUNDED FOLLOWING
```

Therefore:

```text
customer 1 rows
10, 20, 5, 30

every customer-1 row
→ total 65
```

### With orderBy

For an ordered aggregate window such as:

```python
Window \
    .partitionBy("cust") \
    .orderBy("day")
```

the default frame is a growing `RANGE` frame:

```text
RANGE BETWEEN
UNBOUNDED PRECEDING
AND CURRENT ROW
```

We verified this experimentally:

```text
default frame == explicit RANGE frame
→ True
```

---

## 4. RANGE Frame

`RANGE` works from the `ORDER BY` values.

Our customer 1 data contained:

```text
day  amount
1    10
2     5
2    20
3    30
```

There are two rows where:

```text
day = 2
```

They are peers because the window is ordered only by:

```python
orderBy("day")
```

For either `day=2` row, the RANGE frame contains:

```text
(day=1, amount=10)
(day=2, amount=5)
(day=2, amount=20)
```

Therefore:

```text
10 + 5 + 20 = 35
```

Both tied rows receive:

```text
(day=2, amount=5)  → 35
(day=2, amount=20) → 35
```

Observed:

```text
(1,1,10,10)
(1,2,5,35)
(1,2,20,35)
(1,3,30,65)

(2,1,100,100)
(2,2,50,150)
```

---

## 5. ROWS Frame

`ROWS` works from row positions rather than peer values.

Using:

```python
Window \
    .partitionBy("cust") \
    .orderBy("day") \
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
```

makes the frame grow one physical row at a time.

Observed for the tied `day=2` rows:

```text
(day=2, amount=5)  → 35
(day=2, amount=20) → 30
```

This initially looks unusual because the displayed output was later sorted by:

```text
day, amount
```

but the window itself was ordered only by:

```text
day
```

Spark was therefore free to process the two `day=2` rows in either order.

The observed sequence was effectively:

```text
10
→ 10

20
→ 10 + 20 = 30

5
→ 10 + 20 + 5 = 35
```

The display order does not change the order Spark used when evaluating the window.

This proves an important rule:

```text
ROWS + non-unique ORDER BY
→ sequence between tied rows is not deterministic
```

---

## 6. Deterministic ROWS Ordering

We then changed the ordering to:

```python
orderBy(
    "day",
    "amount"
)
```

In our controlled data this uniquely orders the rows.

Customer 1 becomes:

```text
day 1, amount 10
day 2, amount 5
day 2, amount 20
day 3, amount 30
```

Observed:

```text
(day=1, amount=10) → 10

(day=2, amount=5)
10 + 5
→ 15

(day=2, amount=20)
10 + 5 + 20
→ 35

(day=3, amount=30)
10 + 5 + 20 + 30
→ 65
```

Therefore:

```text
ROWS + unique meaningful ordering
→ deterministic sequence
```

In production, the tie-breaker should represent a real business sequence, such as:

```text
transaction_timestamp
transaction_id
version
event_sequence
```

rather than adding an arbitrary column only to force uniqueness.

---

## 7. RANGE vs ROWS

The core difference is:

```text
RANGE
→ based on ORDER BY values

ROWS
→ based on row positions
```

For our tied `day=2` rows:

```text
RANGE

amount 5  → 35
amount 20 → 35
```

because both rows are peers.

With deterministic `ROWS`:

```text
ROWS

amount 5  → 15
amount 20 → 35
```

because the frame advances one row at a time.

### Easy rule

```text
RANGE
→ think "values / peers"

ROWS
→ think "positions / sequence"
```

---

## 8. No partitionBy

When `partitionBy` is omitted:

```python
Window.orderBy("day")
```

all rows belong to one logical window partition.

In our workspace Spark warned:

```text
Moving all data to a single partition
```

That can become a major scalability problem on large datasets.

Our text check:

```text
"SinglePartition" in explain output
```

returned:

```text
False
```

so that specific plan-string check was not useful in this environment.

The observed Spark warning is the evidence we have from this run.

---

## 9. Observed in Our Workspace

Environment:

```text
Spark   : 4.2.0
Compute : Databricks Serverless
```

Controlled data:

```text
source rows  = 6
groupBy rows = 2
window rows  = 6

customer totals:
(1, 65)
(2, 150)
```

Default ordered frame:

```text
(1,1,10,10)
(1,2,5,35)
(1,2,20,35)
(1,3,30,65)
(2,1,100,100)
(2,2,50,150)
```

Validation:

```text
default frame == explicit RANGE frame
→ True
```

ROWS using only `orderBy("day")`:

```text
(1,2,5)  → 35
(1,2,20) → 30
```

The tied-row sequence was not deterministic.

ROWS using:

```python
orderBy("day", "amount")
```

produced:

```text
(1,2,5)  → 15
(1,2,20) → 35
```

with deterministic ordering for this controlled dataset.

---

## 10. Real Data Validation — samples.tpch

For three customers, the window used:

```text
partitionBy:
o_custkey

orderBy:
o_orderdate
o_orderkey

frame:
ROWS BETWEEN
UNBOUNDED PRECEDING
AND CURRENT ROW
```

`o_orderkey` provided a deterministic tie-breaker when multiple orders had the same order date.

On the final ordered row for every tested customer:

```text
running total == full customer total
```

Observed totals:

```text
299701   7,128,450.99
382414   7,139,342.74
721180   7,147,854.45
```

All three validations returned:

```text
True
```

---

## 11. Not Verified

The following were intentionally not completed in S4.6:

```text
row_number()
rank()
dense_rank()
ntile()
```

These belong to **S4.7 — Ranking**.

Also not treated as full topics here:

```text
lag / lead / first / last
→ S4.8

running sum / rolling average
→ S4.9

7-day rolling revenue
→ S4.10
```

The running sum in S4.6 was used only to make window-frame behaviour visible.

---

## 12. Interview Q&A

### Q1. What is the difference between groupBy and a window?

`groupBy` collapses rows to a new aggregate grain.

A window keeps the original rows and calculates values across related rows.

---

### Q2. What are the three main parts of a window specification?

```text
partitionBy
orderBy
frame
```

They answer:

```text
Which rows belong together?
In what order?
Which rows participate in the current calculation?
```

---

### Q3. What is the default frame when orderBy is specified for an aggregate window?

```text
RANGE BETWEEN
UNBOUNDED PRECEDING
AND CURRENT ROW
```

Our explicit RANGE calculation matched the default result.

---

### Q4. What is the difference between RANGE and ROWS?

`RANGE` works from the values of the ordering expression.

`ROWS` works from row positions.

---

### Q5. Why did both day=2 rows receive 35 with RANGE?

They shared the same `day=2` ORDER BY value and were peers in the RANGE frame.

---

### Q6. Why did the two day=2 rows receive different values with ROWS?

ROWS advanced one physical row at a time.

Because `day` did not uniquely order those rows, their relative processing sequence was not guaranteed.

---

### Q7. Why add a tie-breaker?

To create a deterministic sequence when the calculation depends on row position.

---

### Q8. Why can omitting partitionBy be dangerous?

All rows belong to one logical window partition.

For large datasets this can create a single-partition processing bottleneck.

---

## 13. Core Understanding

```text
Window specification
        |
        +-- partitionBy
        |      → which rows belong together
        |
        +-- orderBy
        |      → sequence inside the partition
        |
        +-- frame
               → which rows participate
                  for the current row
```

And:

```text
RANGE
→ values / peers

ROWS
→ positions / sequence
```

When sequence matters:

```text
non-unique ordering
→ nondeterministic tied-row sequence

meaningful unique ordering
→ deterministic sequence
```

**S4.6 — Window Specification COMPLETED.**